# Stage 8 - the accuracy trap, measured directly

Every other notebook *asserts* the accuracy trap from the duplication rate (CICIoV2024 is
99.75% duplicate rows) and from the gap between accuracy and macro-F1. This notebook
**measures it**, by running the same models under the two cross-validation schemes that
`adversec/experiments/crossval.py` was written for but that nothing has ever called:

* **Scheme B - row-level (leaky).** Stratified K-fold over the *duplicated rows*, exactly
  as a naive pipeline would do it. Copies of the same signature land on both sides of the
  fold boundary, so the model is scored partly on frames it memorised.
* **Scheme A - signature-level (honest).** K-fold over *unique signatures*, with the light
  duplication applied inside each fold's training portion only. No copy of a signature
  ever crosses the boundary.

The gap between them is the accuracy trap, in one number, per model, per dataset.

**Runtime note.** The row-level scheme trains on raw duplicated rows, and CICIoV2024's raw
form is 1.4M of them, so the rows are stratified-subsampled to `ROWLEVEL_MAX_ROWS` first.
A random subsample preserves the duplication *structure* (CIC stays ~99.7% duplicated) while
keeping the run tractable. Lower `ROWLEVEL_MAX_ROWS` if this is too slow.

**Requires the raw data** (unlike notebooks 03-07, which run off `datasets/processed/`),
because the duplicated rows are exactly what strict de-duplication threw away.


In [ ]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd, json
import torch
from sklearn.utils.class_weight import compute_class_weight
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN
from adversec.datasets.registry import get_dataset
from adversec.pipeline import audit_duplication, strict_dedup, encode_labels, scale_features
from adversec.experiments.crossval import crossval_rowlevel, crossval_signature_level
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEVICE)

DATASETS = ['ciciov2024', 'road']
ROWLEVEL_MAX_ROWS = 50_000   # stratified cap on the leaky scheme (CIC raw is 1.4M rows)
CNN_EPOCHS        = 50       # same as every other notebook

# Folds are matched WITHIN a dataset so the two schemes are like-for-like. CICIoV2024 is
# capped at 2 because spoofing-GAS has only 2 unique signatures -- a 5-fold split over
# signatures cannot place that class in every fold.
N_SPLITS = {'ciciov2024': 2, 'road': 5}

## Load the raw (still-duplicated) tables

This is the only notebook after 01 that needs `datasets/raw/`. If it is absent, this cell
says so and the rest will not run.


In [ ]:
raw = {}
for name in DATASETS:
    try:
        df = get_dataset(name).load()
    except (FileNotFoundError, OSError) as e:
        print(f'{name}: RAW DATA ABSENT -> {e}')
        continue
    strict = strict_dedup(df)
    audit = audit_duplication(df, subset=FEATURES + [LABEL_COLUMN])
    raw[name] = dict(df=df, strict=strict, audit=audit)
    print(f"{name}: {len(df):,} raw rows -> {len(strict):,} unique signatures "
          f"({audit['duplication_rate_pct']:.2f}% duplicated)")

## Scheme B - row-level CV on the duplicated rows (the leaky baseline)

Scaling and label-encoding are fitted on the whole subsample here, not per fold. That is
deliberate: this cell reproduces the *naive* pipeline in full, and global fitting is part
of what makes it naive. Scheme A below does it properly, inside each fold.


In [ ]:
def subsample_rows(df, max_rows, n_splits, seed=config.RANDOM_SEED):
    """Stratified cap on row count, keeping at least n_splits rows of every class so
    StratifiedKFold can still place each class in every fold."""
    if len(df) <= max_rows:
        return df.reset_index(drop=True)
    frac = max_rows / len(df)
    pieces = []
    for _, g in df.groupby(LABEL_COLUMN):
        n = min(len(g), max(n_splits, int(round(len(g) * frac))))
        pieces.append(g.sample(n=n, random_state=seed))
    out = pd.concat(pieces, ignore_index=True)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


leaky = {}
for name in DATASETS:
    if name not in raw:
        continue
    k = N_SPLITS[name]
    cfg = config.load_dataset_config(name)
    sub = subsample_rows(raw[name]['df'], ROWLEVEL_MAX_ROWS, k)
    sub_audit = audit_duplication(sub, subset=FEATURES + [LABEL_COLUMN])
    print(f"\n=== {name}: SCHEME B (row-level, LEAKY) -- {len(sub):,} rows, "
          f"{sub_audit['duplication_rate_pct']:.2f}% duplicated, {k}-fold ===")

    # Naive pipeline: encode + scale over everything, then split.
    y, _, _ = encode_labels(sub, sub)
    X, _, _ = scale_features(sub, sub, FEATURES)
    X = X.astype(np.float32)

    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(y), y=y)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)

    leaky[name] = crossval_rowlevel(X, y, n_splits=k, device=DEVICE,
                                    random_seed=config.RANDOM_SEED, cnn_epochs=CNN_EPOCHS,
                                    class_weights=cw)
    leaky[name]['_rows'] = int(len(sub))
    leaky[name]['_duplication_pct'] = float(sub_audit['duplication_rate_pct'])

## Scheme A - signature-level CV on unique signatures (the honest measurement)

Same models, same folds, same epochs. The only thing that changes is that no copy of a
signature can straddle the fold boundary, and encoding/scaling/duplication happen inside
each fold's training portion.


In [ ]:
honest = {}
for name in DATASETS:
    if name not in raw:
        continue
    k = N_SPLITS[name]
    cfg = config.load_dataset_config(name)
    print(f"\n=== {name}: SCHEME A (signature-level, HONEST) -- "
          f"{len(raw[name]['strict']):,} signatures, {k}-fold ===")
    honest[name] = crossval_signature_level(
        raw[name]['strict'], FEATURES,
        n_splits=k, dup_target=config.DUP_TARGET, device=DEVICE,
        random_seed=config.RANDOM_SEED, cnn_epochs=CNN_EPOCHS,
        use_duplication=True, use_class_weights=bool(cfg.get('cnn_class_weights')),
    )

## The trap, in one table

`leaky - honest` is how much macro-F1 a naive evaluation would have invented.


In [ ]:
def ms(v):
    a = np.array(v, dtype=float)
    return a.mean(), a.std()

print(f"{'dataset':12s}{'model':8s}{'leaky (row-level)':>22}{'honest (signature)':>22}{'inflation':>12}")
print('-' * 76)
for name in DATASETS:
    if name not in leaky or name not in honest:
        continue
    for mkey, mlab in [('rf', 'RF'), ('cnn', 'CNN')]:
        lm, ls = ms(leaky[name][mkey])
        hm, hs = ms(honest[name][mkey])
        print(f"{name:12s}{mlab:8s}{f'{lm:.4f}+/-{ls:.4f}':>22}{f'{hm:.4f}+/-{hs:.4f}':>22}"
              f"{lm - hm:>+12.4f}")
print()
print('inflation = macro-F1 a leaky, row-level evaluation reports over and above the')
print('honest signature-level measurement of the SAME model on the SAME data.')

## Save (`results/<name>_accuracy_trap.json`)

In [ ]:
def ms_full(v):
    a = np.array(v, dtype=float)
    return {'mean': float(a.mean()), 'std': float(a.std()), 'folds': [float(x) for x in a]}

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in DATASETS:
    if name not in leaky or name not in honest:
        continue
    rep = {
        'dataset': name,
        'method': 'leaky_vs_signature_level_cv',
        'description': (
            'The accuracy trap measured directly. Scheme B (row-level) runs stratified K-fold '
            'over the duplicated rows, so copies of a signature straddle the fold boundary and '
            'the model is partly scored on memorised frames; encoding and scaling are fitted '
            'globally, as a naive pipeline would. Scheme A (signature-level) runs K-fold over '
            'unique signatures with light duplication applied inside each training fold only. '
            'Same models, folds and epochs; the difference is the inflation a leaky evaluation '
            'invents.'),
        'n_splits': N_SPLITS[name],
        'cnn_epochs': CNN_EPOCHS,
        'metric': 'macro-F1',
        'raw_duplication': raw[name]['audit'],
        'rowlevel_subsample': {
            'max_rows': ROWLEVEL_MAX_ROWS,
            'rows_used': leaky[name]['_rows'],
            'duplication_pct_after_subsample': leaky[name]['_duplication_pct'],
            'note': ('random subsample preserves the duplication structure; it caps runtime only. '
                     'Scheme A uses all unique signatures.'),
        },
        'scheme_b_rowlevel_leaky': {m: ms_full(leaky[name][m]) for m in ('rf', 'cnn')},
        'scheme_a_signature_level_honest': {m: ms_full(honest[name][m]) for m in ('rf', 'cnn')},
        'inflation_macro_f1': {
            m: round(float(np.mean(leaky[name][m]) - np.mean(honest[name][m])), 4)
            for m in ('rf', 'cnn')
        },
    }
    path = config.RESULTS_DIR / f'{name}_accuracy_trap.json'
    path.write_text(json.dumps(rep, indent=2))
    print('saved ->', path)